In [ ]:
import visualizer as visualizer

ModuleNotFoundError: No module named 'quantum_nav'

In [ ]:
import solvers.DWave_solver as DWave_solver
import map
import pathFormulation
import config.parser as config_parser
import QUBOBuilder
from config.hdf5parser import load_map_from_hdf5

In [ ]:
conf = config_parser.load_config("config/config.yaml", sections=["problems", "penalty_sets", "solver"])
map_hdf5 = load_map_from_hdf5("maps/synthetic/3x3/no_obs3x3_ter.h5")
grid = map.Grid.from_hdf5_data(map_hdf5)
map_conf = conf["problems"]["grid_3x3_default"]
problem = pathFormulation.PathfindingProblem.from_hdf5_data(map_hdf5, map_conf)

In [ ]:
material_costs = {
    "ceramic": 1.0,
    "water": 5.0,
    "asphalt": 1.5,
    "grass": 2.0
}
penalties_conf = conf["penalty_sets"]["ter"]
QUBOBuilder = QUBOBuilder.QUBOBuilder(problem, penalties=penalties_conf, name="standard", material_cost=material_costs)

In [ ]:
solver = DWave_solver.QUBOSolver(normalize_scale=2.0, num_reads=15)

QUBOBuilder.reset_problem()
QUBOBuilder.build()
solution = solver.solve_qubo(QUBOBuilder)
path = solver.decode_path(solution["solution"], problem)
print("Solution:", solution["solution"])
print("Path:", path)
print(f"Energy: {solver.total_energy(solution):.4f}")

Start position: (2, 0) Iteration: 0
Solution: [{0: np.int8(0), 1: np.int8(0), 2: np.int8(0), 3: np.int8(0), 4: np.int8(0), 5: np.int8(0), 6: np.int8(1), 7: np.int8(0), 8: np.int8(0), 9: np.int8(0), 10: np.int8(0), 11: np.int8(0), 12: np.int8(0), 13: np.int8(0), 14: np.int8(0), 15: np.int8(0), 16: np.int8(1), 17: np.int8(0), 18: np.int8(0), 19: np.int8(0), 20: np.int8(0), 21: np.int8(0), 22: np.int8(0), 23: np.int8(0), 24: np.int8(0), 25: np.int8(0), 26: np.int8(1), 27: np.int8(0), 28: np.int8(0), 29: np.int8(0), 30: np.int8(0), 31: np.int8(0), 32: np.int8(1), 33: np.int8(0), 34: np.int8(0), 35: np.int8(0), 36: np.int8(0), 37: np.int8(0), 38: np.int8(1), 39: np.int8(0), 40: np.int8(0), 41: np.int8(0), 42: np.int8(0), 43: np.int8(0), 44: np.int8(0), 45: np.int8(0), 46: np.int8(0), 47: np.int8(1), 48: np.int8(0), 49: np.int8(0), 50: np.int8(0), 51: np.int8(0), 52: np.int8(0), 53: np.int8(0)}]
Path: [(2, 0, 0), (2, 1, 1), (2, 2, 2), (1, 2, 3), (0, 2, 4), (0, 2, 5)]
Energy: -5.7689


In [ ]:
viz = visualizer.QuantumRoboticsVisualizer()
anim = viz.animate_pathfinding(grid.obstacles, grid.N, grid.M, path, problem.start, problem.end,
                            title="QUBO Solution with Time Steps", jupyter_mode=True)
anim